# Experiment 1: Parallelised-Formulation Comparison Across Datasets

## George Bird, 2026

*Compares four multilayer perceptron formulations across MNIST, CIFAR-10 and Caltech256:*

- **Standard MLP:** `NormalLinear` with `ElementwiseTanh`.
- **Ordinary isotropic MLP:** `NormalLinear` with `IsotropicTanh`.
- **Factorised isotropic MLP:** `FactorisedParallelLinear_FinalOptimised`, implementing the weight-chain factorisation of Eq. 12.
- **Reparameterised isotropic MLP:** `ReparameterisedParallelLinear_MatrixBiases`, implementing the generalised architecture of Sec. 3.1.

The ordinary and factorised isotropic architectures are mathematically equivalent formulations, whereas the reparameterised architecture introduces independently trainable weight and bias parameters, expanding the functional class.

**Architecture:** `[n, 500, 500, 500, 500, m]`, where input and output dimensions depend on the dataset.

**Training:** Adam optimiser, learning rate `1e-3`, batch size `48`, orthogonal weight initialisation, zero-bias initialisation and cross-entropy loss. Models are trained for 100 epochs using consistent random seeds across formulations within each repeat.

**Repeats:** 8 independent repeats per dataset–formulation combination.

**Evaluation:** Test accuracy and cross-entropy loss are recorded throughout training. Results are aggregated across repeats to compare classification performance and training dynamics.

**Paper correspondence:** Section 4, Figure 1 and Appendix A.1 of *Parallelising Matrix-Products in Isotropic Multilayer Perceptrons*.

In [ ]:
import torch
import torch.nn as nn
import gc
import numpy as np
import pandas as pd
import pickle as pkl
import matplotlib as mpl
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os
import time
import math
import random
from Dependency import *
torch.set_float32_matmul_precision('high')

In [ ]:
# Hyperparameters
LEARNING_RATE = 1e-3
BATCH_SIZE = 48
DEVICE = try_gpu(output=True, i=0)
TOTAL_EPOCHS = 100
REPEAT = 8
DATA_ROOT = "./data"
HIDDEN_WIDTHS =  [500, 500, 500, 500] # n and m vary per dataset, so only the hidden portion is fixed here
SAVE_DIR = "./Saved_Models/Experiment 1 - 500x4/"

RANDOM_SEED_BASE = 1234
DATA_SPLIT_SEED = 1234
CALTECH_TEST_FRACTION = 0.2

MODEL_CLASSES = {
    "NormalElementwiseTanhMLP": NormalLinear,
    "NormalIsotropicMLP": NormalLinear,
    "FactorisedParallelMLP": FactorisedParallelLinear_FinalOptimised,
    "ReparameterisedParallelMLP": ReparameterisedParallelLinear_MatrixBiases,
}

os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")
print(f"Hidden widths: {HIDDEN_WIDTHS}")
print(f"Total epochs: {TOTAL_EPOCHS}")
print(f"Repeats per (dataset, formulation): {REPEAT}")

In [ ]:
class MNISTPerPixelNormalize:
    def __init__(self):
        normaliser_dictionary = pkl.load(open("./MNIST_normalisations.pkl", "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        return (tensor - self.mean) * self.inv_std


class CIFARPerPixelNormalize:
    def __init__(self):
        normaliser_dictionary = pkl.load(open("./CIFAR_normalisations.pkl", "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        return (tensor - self.mean) * self.inv_std


class Caltech256PerPixelNormalize:
    def __init__(self):
        normaliser_dictionary = pkl.load(open("./CALTECH256_64x64_normalisations.pkl", "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        return (tensor - self.mean) * self.inv_std


class EnsureRGB:
    """Converts single-channel Caltech256 images to three-channel RGB before tensor conversion, so every sample matches the normalisation's channel count."""
    def __call__(self, image):
        return image.convert("RGB")


class Flatten:
    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        return tensor.reshape(-1)

In [ ]:
mnist_transform = transforms.Compose([transforms.ToTensor(), MNISTPerPixelNormalize(), Flatten()])
cifar_transform = transforms.Compose([transforms.ToTensor(), CIFARPerPixelNormalize(), Flatten()])
caltech_transform = transforms.Compose([
    EnsureRGB(),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    Caltech256PerPixelNormalize(),
    Flatten(),
])

mnist_train = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=mnist_transform)

cifar_train = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=cifar_transform)
cifar_test = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=cifar_transform)

# Caltech256 has no canonical train/test split, so a fixed random split is used instead,
# shared across every repeat and formulation for a fair comparison.
caltech_full = datasets.Caltech256(root=DATA_ROOT, download=True, transform=caltech_transform)

if hasattr(caltech_full, "y"):
    caltech_targets = caltech_full.y
elif hasattr(caltech_full, "targets"):
    caltech_targets = caltech_full.targets
else:
    raise AttributeError("Caltech256 dataset exposes neither \"y\" nor \"targets\" for its labels")

CALTECH_CLASSES = len(set(caltech_targets))

caltech_test_size = int(len(caltech_full) * CALTECH_TEST_FRACTION)
caltech_train_size = len(caltech_full) - caltech_test_size
caltech_train, caltech_test = random_split(
    caltech_full,
    [caltech_train_size, caltech_test_size],
    generator=torch.Generator().manual_seed(DATA_SPLIT_SEED),
)

DATASETS = {
    "MNIST": (
        DataLoader(mnist_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
        DataLoader(mnist_test, batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
        28 * 28 * 1,
        10,
    ),
    "CIFAR10": (
        DataLoader(cifar_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
        DataLoader(cifar_test, batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
        32 * 32 * 3,
        10,
    ),
    "Caltech256": (
        DataLoader(caltech_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
        DataLoader(caltech_test, batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
        64 * 64 * 3,
        CALTECH_CLASSES,
    ),
}

print("Dataset sizes:")
for dataset_name, (train_loader, test_loader, input_width, output_width) in DATASETS.items():
    print(f"{dataset_name}: train={len(train_loader.dataset)}, test={len(test_loader.dataset)}, n={input_width}, m={output_width}")

In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def repeat_history_path(dataset_name: str, model_name: str, repeat_idx: int) -> str:
    return os.path.join(SAVE_DIR, f"{dataset_name}_{model_name}_REPEAT_{repeat_idx}.pkl")


def load_all_histories(dataset_name: str, model_name: str, repeats: int) -> list[dict]:
    histories = []
    for repeat_idx in range(1, repeats + 1):
        with open(repeat_history_path(dataset_name, model_name, repeat_idx), "rb") as f:
            histories.append(pkl.load(f))
    return histories

In [ ]:
FORCE_RERUN = False

# Note: training_loop() prints a blank line at the end of every epoch regardless of `quiet`,
# so even with quiet=True the 90 runs below will still produce plenty of console output.

for dataset_name, (train_loader, test_loader, input_width, output_width) in DATASETS.items():
    width_structure = [input_width] + HIDDEN_WIDTHS + [output_width]

    for model_name, model_class in MODEL_CLASSES.items():
        for repeat in range(REPEAT):
            repeat_idx = repeat + 1
            save_path = repeat_history_path(dataset_name, model_name, repeat_idx)

            if os.path.exists(save_path) and not FORCE_RERUN:
                print(f"Skipping {dataset_name} | {model_name} | Repeat {repeat_idx}/{REPEAT}...")
                continue

            print(f"\n\n========== {dataset_name} | {model_name} | Repeat {repeat_idx}/{REPEAT} | Architecture {width_structure} ==========")

            set_global_seed(RANDOM_SEED_BASE + repeat_idx)

            if "NormalElementwiseTanhMLP" == model_name:
                print("STANDARD TANH ACTIVATION FUNCTION:")
                network = model_class(width_structure, ElementwiseTanh)
            else:
                print("STANDARD ISOTROPIC TANH ACTIVATION FUNCTION:")
                network = model_class(width_structure, IsotropicTanh)

            network.reset_parameters()
            network.to(DEVICE)

            compiled_network = torch.compile(
                network,
                # mode=""
            )

            start_time = time.time()

            _, history = training_loop(
                network=compiled_network,
                training_set=train_loader,
                testing_set=test_loader,
                epochs=TOTAL_EPOCHS,
                learning_rate=LEARNING_RATE,
                device=DEVICE,
                quiet=False,
                classification_or_reconstruction="classification",
            )

            elapsed_seconds = time.time() - start_time

            network.cpu()

            train_x, test_x, train_cost, test_cost, train_acc, test_acc = history

            print(
                f"Finished in {elapsed_seconds:6.1f}s | "
                f"Final test accuracy: {test_acc[-1]:5.2f}%"
            )

            repeat_history = {
                "dataset": dataset_name,
                "model": model_name,
                "repeat_idx": repeat_idx,
                "width_structure": width_structure,
                "hyperparameters": {
                    "learning_rate": LEARNING_RATE,
                    "batch_size": BATCH_SIZE,
                    "total_epochs": TOTAL_EPOCHS,
                },
                "final_state_dict": {
                    k: v.detach().cpu().clone()
                    for k, v in network.state_dict().items()
                },
                "train_x": train_x,
                "train_cost": train_cost,
                "test_x": test_x,
                "test_cost": test_cost,
                "train_acc": train_acc,
                "test_acc": test_acc,
                "elapsed_seconds": elapsed_seconds,
            }

            with open(save_path, "wb") as f:
                pkl.dump(repeat_history, f)

            del compiled_network
            del network
            gc.collect()
            torch.compiler.reset()
            torch.cuda.empty_cache()

## Results

Loads every saved repeat history and aggregates mean ± SEM test accuracy and test cost, per dataset and formulation, at the final epoch.

In [ ]:
rows = []
for dataset_name in DATASETS:
    for model_name in MODEL_CLASSES:
        histories = load_all_histories(dataset_name, model_name, REPEAT)

        final_test_acc = np.asarray([h["test_acc"][-1] for h in histories], dtype=np.float64)
        final_test_cost = np.asarray([h["test_cost"][-1] for h in histories], dtype=np.float64)

        rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "test_acc_mean": final_test_acc.mean(),
            "test_acc_sem": final_test_acc.std(ddof=1) / math.sqrt(len(histories)),
            "test_cost_mean": final_test_cost.mean(),
            "test_cost_sem": final_test_cost.std(ddof=1) / math.sqrt(len(histories)),
            "n": len(histories),
        })

summary = pd.DataFrame(rows)

summary["test accuracy (%)"] = summary.apply(
    lambda r: f"{r['test_acc_mean']:.2f} ± {r['test_acc_sem']:.2f}", axis=1
)
summary["test cost"] = summary.apply(
    lambda r: f"{r['test_cost_mean']:.4f} ± {r['test_cost_sem']:.4f}", axis=1
)

summary[["dataset", "model", "test accuracy (%)", "test cost", "n"]]

In [ ]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "grid.linewidth": 0.5,
    "grid.alpha": 0.22,
    "lines.linewidth": 1.6,
})

MODEL_COLOURS = {
    "NormalElementwiseTanhMLP": "tab:blue",
    "NormalIsotropicMLP": "tab:orange",
    "FactorisedParallelMLP": "tab:green",
    "ReparameterisedParallelMLP": "tab:red",
}


def mean_std(arr_2d: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    means = arr_2d.mean(axis=0)
    stds = arr_2d.std(axis=0, ddof=1) if arr_2d.shape[0] > 1 else np.zeros(arr_2d.shape[1])
    return means, stds


os.makedirs("./TempImages", exist_ok=True)

fig, axes = plt.subplots(1, len(DATASETS), figsize=(5.5 * len(DATASETS), 3.5), constrained_layout=True)

for i, (ax, dataset_name) in enumerate(zip(axes, DATASETS)):
    for model_name in MODEL_CLASSES:
        print(dataset_name, model_name)
        histories = load_all_histories(dataset_name, model_name, REPEAT)
        test_acc_2d = np.asarray([h["test_acc"] for h in histories], dtype=np.float64)
        epochs = np.asarray(histories[0]["test_x"], dtype=np.float64)

        means, stds = mean_std(test_acc_2d)
        colour = MODEL_COLOURS[model_name]
        ax.plot(epochs, means, color=colour, label=model_name, alpha=0.5)
        ax.fill_between(epochs, means - stds, means + stds, color=colour, alpha=0.15, linewidth=0)

    ax.set_title(dataset_name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Test Accuracy (%)")
    ax.grid(True, which="major", axis="both")
    if i == 0: ax.set_ylim(90, 100)
    elif i == 1: ax.set_ylim(40,48)
    elif i == 2: ax.set_ylim(11, 17)
axes[0].legend(frameon=False, loc="lower right")
fig.savefig(f"./TempImages/Experiment1_TestAccuracy{','.join([str(p) for p in HIDDEN_WIDTHS])}.svg", bbox_inches="tight")
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Publication-style plotting defaults
# ---------------------------------------------------------------------

mpl.rcParams.update({
    # Explicit light theme
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",

    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    # Typography
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,

    # Axes
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "axes.axisbelow": True,

    # Lines
    "lines.linewidth": 2.0,

    # Ticks
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,

    # Grid
    "grid.color": "0.75",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.35,

    # Saving
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})


# ---------------------------------------------------------------------
# Model styles
# Colourblind-safe colours + different line styles
# ---------------------------------------------------------------------

MODEL_STYLES = {
    "NormalElementwiseTanhMLP": {
        "label": "Standard Elementwise",
        "color": "#0072B2",
        "linestyle": "-",
    },
    "NormalIsotropicMLP": {
        "label": "Standard Isotropic",
        "color": "#E69F00",
        "linestyle": "--",
    },
    "FactorisedParallelMLP": {
        "label": "Factorised Isotropic",
        "color": "#009E73",
        "linestyle": "-.",
    },
    "ReparameterisedParallelMLP": {
        "label": "Reparameterised Isotropic",
        "color": "#D55E00",
        "linestyle": ":",
    },
}


def mean_std(arr_2d: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    means = arr_2d.mean(axis=0)

    if arr_2d.shape[0] > 1:
        stds = arr_2d.std(axis=0, ddof=1)
    else:
        stds = np.zeros(arr_2d.shape[1])

    return means, stds


# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------

os.makedirs("./TempImages", exist_ok=True)

n_datasets = len(DATASETS)

fig, axes = plt.subplots(
    1,
    n_datasets,
    figsize=(4 * n_datasets, 3.4),
    constrained_layout=True,
    facecolor="white",
)

axes = np.atleast_1d(axes)

legend_handles = []

for i, (ax, dataset_name) in enumerate(zip(axes, DATASETS)):

    ax.set_facecolor("white")

    for model_name in MODEL_CLASSES:
        print(dataset_name, model_name)

        histories = load_all_histories(
            dataset_name,
            model_name,
            REPEAT,
        )

        test_acc_2d = np.asarray(
            [h["test_acc"] for h in histories],
            dtype=np.float64,
        )

        epochs = np.asarray(
            histories[0]["test_x"],
            dtype=np.float64,
        )

        means, stds = mean_std(test_acc_2d)
        style = MODEL_STYLES[model_name]

        line, = ax.plot(
            epochs,
            means,
            color=style["color"],
            linestyle=style["linestyle"],
            label=style["label"],
            linewidth=2.0,
            zorder=3,
        )

        ax.fill_between(
            epochs,
            means - stds,
            means + stds,
            color=style["color"],
            alpha=0.12,
            linewidth=0,
            zorder=2,
        )

        if i == 0:
            legend_handles.append(line)

    ax.set_title(
        dataset_name,
        pad=7,
        fontweight="semibold",
    )

    ax.set_xlabel("Epoch")

    # Horizontal grid only
    ax.grid(
        True,
        which="major",
        axis="y",
    )

    ax.tick_params(
        axis="both",
        direction="out",
    )

    # Dataset-specific y limits
    if i == 0:
        ax.set_ylim(90, 100)
    elif i == 1:
        ax.set_ylim(40, 48)
    elif i == 2:
        ax.set_ylim(11, 17)
    # ax.set_xlim(0, 40)

# Shared y label
fig.supylabel(
    "Test accuracy (%)",
    fontsize=10,
)


# Shared legend
fig.legend(
    handles=legend_handles,
    loc="outside upper center",
    ncol=len(legend_handles),
    frameon=False,
    handlelength=2.8,
    columnspacing=1.6,
    handletextpad=0.6,
)


# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------

hidden_width_string = ",".join(map(str, HIDDEN_WIDTHS))

output_path = (
    "./TempImages/"
    f"Experiment1_TestAccuracy{hidden_width_string}.svg"
)

fig.savefig(
    output_path,
    facecolor="white",
    bbox_inches="tight",
)

plt.show()